# Physics Applications

Use this notebook when your starting point is a **physical model or a three-dimensional
physical surface** and you want to turn that geometry into a spatial graph and then
study its topology.

You do **not** need to run every section. A new user should normally follow this order:

1. build one `NodalSkeleton`;
2. inspect the exceptional surface;
3. inspect the skeleton points;
4. inspect the embedded spatial graph;
5. inspect any physical vector/scalar fields of interest;
6. select a planar projection;
7. inspect the PD code; and
8. compute the Yamada polynomial only after the geometry looks correct.

The main worked model and the model gallery use **`dimension=300`**. A few targeted
custom-model and parameter-scan examples deliberately use `dimension=200` to keep those
secondary demonstrations lighter; each such code cell states its sampling resolution explicitly.

### Objects you will see repeatedly

- **`NodalSkeleton`**: samples the model on a 3D momentum grid and provides surface,
  skeleton, graph, and field data.
- **Exceptional surface**: the continuous three-dimensional geometry from which the
  graph is extracted.
- **Skeleton points**: the discrete medial-axis points before they are reduced to a graph.
- **Embedded graph**: a `networkx.MultiGraph` whose nodes carry 3D positions and whose
  edges carry 3D polylines.
- **Planar projection / PD code**: a two-dimensional diagram of the embedded graph with
  crossing information.
- **Yamada polynomial**: the invariant computed from the trusted projected graph.

If you are not yet comfortable with the surface → graph → projection pipeline, first run
**Getting Started** and **Core Workflows**.

## 1. Set up the physics examples

Run the setup and import cells once.

Most sections below are independent after setup, so you can jump directly to the
physical example you need. The nodal/surface examples require the optional scientific
dependencies used by `knotted_graph.applications.nodal`.

In [ ]:
from pathlib import Path
import sys
import os
import tempfile
import importlib.util

PROJECT_ROOT = Path.cwd().resolve()
while (
    not (PROJECT_ROOT / "src").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(
        "Could not locate the KnottedGraph repository root. "
        "Run this notebook from inside the repository checkout."
    )

SRC_ROOT = PROJECT_ROOT / "src"
DOC_ASSETS = PROJECT_ROOT / "doc" / "assets"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "knottedgraph-mpl"),
)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
for package in [
    "numpy", "sympy", "networkx", "matplotlib",
    "plotly", "pyvista", "skimage", "poly2graph",
]:
    print(f"{package:12s} = {importlib.util.find_spec(package) is not None}")

In [2]:
import numpy as np
import sympy as sp
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from knotted_graph.applications.nodal import NodalSkeleton
from knotted_graph.applications.nodal.models import (
    unknot_bloch_vector,
    hopf_link_bloch_vector,
    trefoil_bloch_vector,
    solomon_bloch_vector,
    threelink_bloch_vector,
    awesome_bloch_vector,
    pq_torus_knot_bloch_vector,
)
from knotted_graph.projection import (
    compute_yamada_polynomial,
    select_projection,
    PDCode,
)
from knotted_graph.visualization import plot_3D_graph_plotly

Y = sp.Symbol("Y")
kx, ky, kz = sp.symbols("k_x k_y k_z", real=True)

PHYSICS_CAMERA = dict(
    eye=dict(
        x=4.0,
        y=4.0,
        z=3.0,
    )
)


def mesh_trace(surface, *, opacity=0.35, name="surface"):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = np.asarray(mesh.points)
    return go.Mesh3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        opacity=opacity,
        name=name,
        showscale=False,
    )


def clean_scene(fig, *, title=None):
    fig.update_layout(
        title=title,
        showlegend=False,
        scene=dict(
            xaxis=dict(title="k_x", showbackground=False),
            yaxis=dict(title="k_y", showbackground=False),
            zaxis=dict(title="k_z", showbackground=False),
            aspectmode="data",
            camera=PHYSICS_CAMERA,
        ),
        margin=dict(l=0, r=0, b=0, t=35 if title else 5),
    )
    return fig


def show_surface(surface, *, title=None, opacity=0.5):
    return clean_scene(
        go.Figure(mesh_trace(surface, opacity=opacity)),
        title=title,
    )


def show_skeleton_points(points, *, title=None):
    pts = np.asarray(points)
    fig = go.Figure(
        go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode="markers",
            marker=dict(size=2.5),
        )
    )
    return clean_scene(fig, title=title)


def show_graph(graph, *, title=None):
    return clean_scene(plot_3D_graph_plotly(graph), title=title)


BLUE = "#1f77b4"
RED = "#d62728"

def plot_projection_diagram(
    projection,
    *,
    title=None,
    annotate=False,
    figsize=(5.4, 4.6),
    edge_color=BLUE,
    vertex_color=RED,
    line_width=2.4,
    vertex_size=64,
    gap_fraction=0.015,
):
    """Draw blue edges, red vertices, and explicit over/under crossing gaps."""
    fig, ax = plt.subplots(figsize=figsize)

    arcs_by_id = {
        arc.id: arc
        for arc in projection.arcs
    }

    all_xy = []

    # Draw the complete projected diagram first.
    for arc in projection.arcs:
        x, y = arc.line.xy
        ax.plot(
            x,
            y,
            color=edge_color,
            linewidth=line_width,
            solid_capstyle="round",
            zorder=1,
        )
        all_xy.extend(zip(x, y))

        if annotate:
            midpoint = arc.line.interpolate(
                0.5,
                normalized=True,
            )
            ax.text(
                midpoint.x,
                midpoint.y,
                f"a{arc.id}",
                fontsize=8,
                color=edge_color,
                zorder=7,
            )

    if all_xy:
        xy = np.asarray(
            all_xy,
            dtype=float,
        )
        diagram_span = max(
            float(np.ptp(xy[:, 0])),
            float(np.ptp(xy[:, 1])),
            1.0,
        )
    else:
        diagram_span = 1.0

    gap_radius = (
        gap_fraction
        * diagram_span
    )

    for crossing in projection.crossings:
        try:
            ordered_arcs = list(
                crossing.ccw_ordered_arcs
            )
        except Exception:
            ordered_arcs = []

        if len(ordered_arcs) != 4:
            continue

        cx = crossing.point.x
        cy = crossing.point.y

        # Temporarily erase both strands around the crossing.
        ax.add_patch(
            plt.Circle(
                (cx, cy),
                gap_radius,
                facecolor="white",
                edgecolor="none",
                zorder=4,
            )
        )

        # In KnottedGraph's crossing order, entries 0 and 2 form
        # the over-strand. Redraw those two half-arcs continuously.
        for arc_id in (
            ordered_arcs[0],
            ordered_arcs[2],
        ):
            arc = arcs_by_id.get(
                arc_id
            )

            if (
                arc is None
                or arc.line.length <= 0
            ):
                continue

            probe_distance = min(
                2.4 * gap_radius,
                0.45 * arc.line.length,
            )

            if (
                arc.start_type == "x"
                and arc.start_id
                == crossing.id
            ):
                outside_point = (
                    arc.line.interpolate(
                        probe_distance
                    )
                )

            elif (
                arc.end_type == "x"
                and arc.end_id
                == crossing.id
            ):
                outside_point = (
                    arc.line.interpolate(
                        max(
                            arc.line.length
                            - probe_distance,
                            0.0,
                        )
                    )
                )

            else:
                continue

            ax.plot(
                [cx, outside_point.x],
                [cy, outside_point.y],
                color=edge_color,
                linewidth=line_width,
                solid_capstyle="round",
                zorder=5,
            )

        if annotate:
            ax.annotate(
                f"x{crossing.id}",
                (cx, cy),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
                color="black",
                zorder=8,
            )

    # Rigid graph vertices are always red.
    for vertex in projection.vertices:
        ax.scatter(
            *vertex.point.xy,
            color=vertex_color,
            s=vertex_size,
            zorder=6,
        )

        if annotate:
            ax.annotate(
                f"v{vertex.id}",
                vertex.point.xy,
                xytext=(5, -10),
                textcoords="offset points",
                fontsize=8,
                color=vertex_color,
                zorder=8,
            )

    ax.set_aspect("equal")
    ax.axis("off")

    if title:
        ax.set_title(title)

    plt.show()
    return fig, ax


def show_projection(
    projection,
    *,
    title=None,
):
    """Application-level alias for the common projection renderer."""
    return plot_projection_diagram(
        projection,
        title=title,
    )

### 1.1 Shared plotting helpers

In [3]:
def add_surface_trace(fig, surface, *, row=1, col=1, opacity=0.58, color=BLUE):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = np.asarray(mesh.points)
    fig.add_trace(
        go.Mesh3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            color=color, opacity=opacity, showscale=False,
        ),
        row=row, col=col,
    )


def add_points_trace(fig, points, *, row=1, col=1, size=2.5, color=BLUE):
    points = np.asarray(points)
    fig.add_trace(
        go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            mode="markers",
            marker=dict(size=size, color=color),
            showlegend=False,
        ),
        row=row, col=col,
    )


def add_graph_traces(fig, graph, *, row=1, col=1):
    graph_fig = plot_3D_graph_plotly(graph)
    for trace in graph_fig.data:
        fig.add_trace(trace, row=row, col=col)


def style_plotly_scenes(fig, scene_count, *, width=980, height=660):
    for index in range(1, scene_count + 1):
        scene_name = "scene" if index == 1 else f"scene{index}"
        fig.update_layout(**{
            scene_name: dict(
                xaxis=dict(title="", showticklabels=False, showbackground=False),
                yaxis=dict(title="", showticklabels=False, showbackground=False),
                zaxis=dict(title="", showticklabels=False, showbackground=False),
                aspectmode="data",
                camera=MATERIAL_CAMERA,
            )
        })
    fig.update_layout(
        width=width, height=height,
        margin=dict(l=0, r=0, t=10, b=0),
        showlegend=False,
    )
    return fig


def polyline_traces_from_slice(polydata):
    points = np.asarray(polydata.points)
    lines = np.asarray(polydata.lines)
    traces = []
    cursor = 0
    while cursor < len(lines):
        n = int(lines[cursor])
        ids = lines[cursor + 1:cursor + 1 + n]
        cursor += n + 1
        if n >= 2:
            traces.append(points[ids])
    return traces


def surface_component_summary(surface):
    """Return the number and sizes of connected surface components."""
    try:
        bodies = surface.split_bodies()
        components = [
            body
            for body in bodies
            if body is not None and getattr(body, "n_cells", 0) > 0
        ]
        sizes = sorted(
            [
                (component.n_points, component.n_cells)
                for component in components
            ],
            key=lambda item: item[1],
            reverse=True,
        )
        return len(components), sizes
    except Exception:
        # Plotting still uses the full surface even if this diagnostic is
        # unavailable in the installed PyVista version.
        return None, []


def print_surface_summary(label, surface):
    """Print mesh size and connected-component information."""
    n_components, component_sizes = surface_component_summary(surface)

    print(
        f"{label}: "
        f"{surface.n_points} points, "
        f"{surface.n_cells} cells"
    )

    if n_components is None:
        print("  connected components = unavailable")
    else:
        print(f"  connected components = {n_components}")
        if n_components > 1:
            print(
                "  component sizes (points, cells) =",
                component_sizes,
            )

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Ellipse, FancyBboxPatch

def _turn_off(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")


def surface_icon(ax, kind, label=None):
    _turn_off(ax)

    if kind == "torus":
        ax.add_patch(Ellipse((0.5, 0.55), 0.72, 0.42, fill=False, linewidth=2))
        ax.add_patch(Ellipse((0.5, 0.55), 0.28, 0.13, fill=False, linewidth=2))
    elif kind == "sphere":
        ax.add_patch(Circle((0.5, 0.55), 0.24, fill=False, linewidth=2))
    elif kind == "octahedral":
        pts = np.array([
            [0.50, 0.82],
            [0.78, 0.55],
            [0.50, 0.28],
            [0.22, 0.55],
        ])
        ax.plot(*pts[[0,1,2,3,0]].T, linewidth=2)
        ax.plot([0.50, 0.50], [0.82, 0.28], linewidth=2)
        ax.plot([0.22, 0.78], [0.55, 0.55], linewidth=2)
    elif kind == "split_pair":
        ax.add_patch(Circle((0.36, 0.55), 0.16, fill=False, linewidth=2))
        ax.add_patch(Circle((0.64, 0.55), 0.16, fill=False, linewidth=2))
    elif kind == "three_component":
        ax.add_patch(Circle((0.30, 0.58), 0.14, fill=False, linewidth=2))
        ax.add_patch(Circle((0.70, 0.58), 0.14, fill=False, linewidth=2))
        ax.add_patch(Circle((0.50, 0.33), 0.14, fill=False, linewidth=2))
    elif kind == "complex_connected":
        t = np.linspace(0, 2*np.pi, 400)
        x = 0.5 + 0.30*np.cos(t) + 0.08*np.cos(3*t)
        y = 0.55 + 0.22*np.sin(t) + 0.06*np.sin(4*t)
        ax.plot(x, y, linewidth=2)
        ax.plot([0.30, 0.70], [0.55, 0.55], linewidth=1.5)
        ax.plot([0.50, 0.50], [0.33, 0.77], linewidth=1.5)
    elif kind == "critical_split":
        ax.add_patch(Ellipse((0.42, 0.55), 0.38, 0.24, fill=False, linewidth=2))
        ax.add_patch(Ellipse((0.58, 0.55), 0.38, 0.24, fill=False, linewidth=2))
    elif kind == "nodal_net":
        pts = np.array([
            [0.25, 0.25],
            [0.75, 0.25],
            [0.75, 0.75],
            [0.25, 0.75],
            [0.25, 0.25],
        ])
        ax.plot(*pts.T, linewidth=2)
        ax.plot([0.25, 0.75], [0.25, 0.75], linewidth=2)
        ax.plot([0.25, 0.75], [0.75, 0.25], linewidth=2)
        ax.plot([0.50, 0.50], [0.20, 0.80], linewidth=2)
        ax.plot([0.20, 0.80], [0.50, 0.50], linewidth=2)
    else:
        ax.text(0.5, 0.5, kind, ha="center", va="center", wrap=True)

    if label:
        ax.text(0.5, 0.06, label, ha="center", va="bottom", wrap=True)


def graph_icon(ax, kind, label=None):
    _turn_off(ax)

    if kind == "B1":
        ax.plot([0.5], [0.5], marker="o", markersize=8)
        t = np.linspace(0, 2*np.pi, 400)
        ax.plot(0.5 + 0.16*np.cos(t), 0.62 + 0.11*np.sin(t), linewidth=2)
    elif kind == "B0":
        ax.plot([0.5], [0.5], marker="o", markersize=8)
    elif kind == "theta6":
        ax.plot([0.25, 0.75], [0.50, 0.50], "o", markersize=8)
        for off in np.linspace(-0.18, 0.18, 6):
            t = np.linspace(0, 1, 200)
            x = 0.25 + 0.50*t
            y = 0.50 + off*np.sin(np.pi*t)
            ax.plot(x, y, linewidth=1.6)
    elif kind == "octahedral_graph":
        pts = {
            "T": (0.50, 0.82),
            "R": (0.78, 0.55),
            "B": (0.50, 0.28),
            "L": (0.22, 0.55),
            "F": (0.50, 0.68),
            "K": (0.50, 0.42),
        }
        edges = [
            ("T","R"),("R","B"),("B","L"),("L","T"),
            ("T","K"),("R","K"),("B","F"),("L","F"),
            ("F","R"),("F","L"),("K","R"),("K","L"),
        ]
        for a, b in edges:
            ax.plot([pts[a][0], pts[b][0]], [pts[a][1], pts[b][1]], linewidth=1.5)
        for x, y in pts.values():
            ax.plot([x], [y], marker="o", markersize=5)
    elif kind == "split_graph_pair":
        ax.plot([0.32, 0.68], [0.5, 0.5], "o", markersize=8)
    elif kind == "multi_component":
        ax.plot([0.25, 0.50, 0.75], [0.55, 0.35, 0.55], "o", markersize=7)
        t = np.linspace(0, 2*np.pi, 200)
        for cx, cy, rx, ry in [(0.25,0.55,0.08,0.06),(0.50,0.35,0.08,0.06),(0.75,0.55,0.08,0.06)]:
            ax.plot(cx + rx*np.cos(t), cy + ry*np.sin(t), linewidth=1.5)
    elif kind == "branched_complex":
        pts = np.array([[0.20, 0.55],[0.40,0.75],[0.60,0.72],[0.78,0.52],[0.60,0.30],[0.36,0.34],[0.22,0.55]])
        ax.plot(pts[:,0], pts[:,1], linewidth=1.8)
        ax.plot([0.36,0.60],[0.34,0.72], linewidth=1.4)
        for x, y in pts[:-1]:
            ax.plot([x], [y], marker="o", markersize=5)
    else:
        ax.text(0.5, 0.5, kind, ha="center", va="center", wrap=True)

    if label:
        ax.text(0.5, 0.06, label, ha="center", va="bottom", wrap=True)


def plot_material_transition_summary(material_name, columns, *, subtitle=None):
    """
    columns: list of dicts with keys:
      window, surface_kind, surface_label, graph_kind, graph_label, invariant
    """
    n = len(columns)
    fig, axes = plt.subplots(
        3, n,
        figsize=(4.1*n, 8.0),
        constrained_layout=True,
    )

    if n == 1:
        axes = np.asarray(axes).reshape(3, 1)

    row_labels = [
        "Energy window",
        "Surface topology",
        "Graph representative",
    ]

    for j, col in enumerate(columns):
        # row 0
        ax = axes[0, j]
        _turn_off(ax)
        box = FancyBboxPatch((0.08, 0.18), 0.84, 0.64, boxstyle="round,pad=0.04", fill=False, linewidth=1.8)
        ax.add_patch(box)
        ax.text(0.5, 0.58, col["window"], ha="center", va="center", fontsize=12, wrap=True)
        ax.text(0.5, 0.30, col["invariant"], ha="center", va="center", fontsize=11, wrap=True)
        if j == 0:
            ax.text(-0.08, 0.50, row_labels[0], ha="right", va="center", fontsize=11, transform=ax.transAxes)

        # row 1
        ax = axes[1, j]
        surface_icon(ax, col["surface_kind"], col.get("surface_label"))
        if j == 0:
            ax.text(-0.08, 0.50, row_labels[1], ha="right", va="center", fontsize=11, transform=ax.transAxes)

        # row 2
        ax = axes[2, j]
        graph_icon(ax, col["graph_kind"], col.get("graph_label"))
        if j == 0:
            ax.text(-0.08, 0.50, row_labels[2], ha="right", va="center", fontsize=11, transform=ax.transAxes)

    fig.suptitle(material_name + (f"\n{subtitle}" if subtitle else ""), fontsize=16)
    plt.show()


def nice_percentile_clip(values, upper=0.98):
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return values, 1.0
    vmax = np.quantile(finite, upper)
    if vmax <= 0:
        vmax = np.max(finite) if finite.size else 1.0
    return np.clip(values, 0, vmax), vmax

In [5]:
MATERIAL_CAMERA = dict(
    eye=dict(
        x=1.65,
        y=1.65,
        z=1.25,
    ),
    center=dict(
        x=0.0,
        y=0.0,
        z=0.0,
    ),
)


def material_axis_style():
    return dict(
        title="",
        showbackground=False,
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        showline=True,
        linewidth=2,
    )


def style_material_figure(
    fig,
    *,
    camera=None,
    width=720,
    height=620,
):
    if camera is None:
        camera = MATERIAL_CAMERA

    fig.update_layout(
        title=None,
        width=width,
        height=height,
        showlegend=False,
        margin=dict(
            l=0,
            r=0,
            t=0,
            b=0,
        ),
        scene=dict(
            xaxis=material_axis_style(),
            yaxis=material_axis_style(),
            zaxis=material_axis_style(),
            aspectmode="data",
            camera=camera,
        ),
    )

    return fig


def show_material_surface(
    surface,
    *,
    opacity=0.50,
):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = np.asarray(mesh.points)

    fig = go.Figure(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=BLUE,
            opacity=opacity,
            showscale=False,
            showlegend=False,
        )
    )

    return style_material_figure(
        fig
    )


def show_material_graph(
    graph,
):
    fig = plot_3D_graph_plotly(
        graph
    )

    return style_material_figure(
        fig
    )


def co2mnga_hamiltonian_sympy():
    """Notebook-local six-band Co2MnGa Hamiltonian used in this example."""
    p = dict(
        t1=-0.31,
        t2=-0.018,
        t3=-0.01,
        t4=0.2,
        t5=-0.02,
        t6=0.04,
        t7=0.28,
        t8=-0.34,
        eps_d=-0.6,
        eps_p=0.6,
    )

    cx2, cy2, cz2 = (
        sp.cos(kx / 2),
        sp.cos(ky / 2),
        sp.cos(kz / 2),
    )
    sx2, sy2, sz2 = (
        sp.sin(kx / 2),
        sp.sin(ky / 2),
        sp.sin(kz / 2),
    )

    xi_d1 = (
        4 * p["t1"] * cx2 * cz2
        + 2 * p["t2"] * (sp.cos(kx) + sp.cos(kz))
        + 2 * p["t3"] * sp.cos(ky)
        + p["eps_d"]
    )
    xi_d2 = (
        4 * p["t1"] * cx2 * cy2
        + 2 * p["t2"] * (sp.cos(kx) + sp.cos(ky))
        + 2 * p["t3"] * sp.cos(kz)
        + p["eps_d"]
    )
    xi_d3 = (
        4 * p["t1"] * cy2 * cz2
        + 2 * p["t2"] * (sp.cos(ky) + sp.cos(kz))
        + 2 * p["t3"] * sp.cos(kx)
        + p["eps_d"]
    )

    xi_p1 = (
        4 * p["t4"] * cy2 * cz2
        + 2 * p["t5"] * (sp.cos(ky) + sp.cos(kz))
        + 2 * p["t6"] * sp.cos(kx)
        + p["eps_p"]
    )
    xi_p2 = (
        4 * p["t4"] * cx2 * cz2
        + 2 * p["t5"] * (sp.cos(kx) + sp.cos(kz))
        + 2 * p["t6"] * sp.cos(ky)
        + p["eps_p"]
    )
    xi_p3 = (
        4 * p["t4"] * cx2 * cy2
        + 2 * p["t5"] * (sp.cos(kx) + sp.cos(ky))
        + 2 * p["t6"] * sp.cos(kz)
        + p["eps_p"]
    )

    xi_p12 = -4 * p["t7"] * sx2 * sy2
    xi_p13 = -4 * p["t7"] * sx2 * sz2
    xi_p23 = -4 * p["t7"] * sy2 * sz2

    xi_dp11 = 2 * p["t8"] * sz2
    xi_dp12 = 0
    xi_dp13 = 2 * p["t8"] * sx2

    xi_dp21 = 2 * p["t8"] * sy2
    xi_dp22 = 2 * p["t8"] * sz2
    xi_dp23 = 0

    xi_dp31 = 0
    xi_dp32 = 2 * p["t8"] * sx2
    xi_dp33 = 2 * p["t8"] * sy2

    return sp.Matrix(
        [
            [xi_d1, 0, 0, xi_dp11, xi_dp12, xi_dp13],
            [0, xi_d2, 0, xi_dp21, xi_dp22, xi_dp23],
            [0, 0, xi_d3, xi_dp31, xi_dp32, xi_dp33],
            [xi_dp11, xi_dp21, xi_dp31, xi_p1, xi_p12, xi_p13],
            [xi_dp12, xi_dp22, xi_dp32, xi_p12, xi_p2, xi_p23],
            [xi_dp13, xi_dp23, xi_dp33, xi_p13, xi_p23, xi_p3],
        ]
    )

## 2. Inspect the physical data before computing topology

A `NodalSkeleton` contains much more than a final graph. Before asking whether two
graphs are topologically different, inspect the physical object that produced the graph.

In this first example you will examine, in order:

1. the exceptional surface;
2. the medial-axis skeleton points;
3. the simplified embedded graph; and
4. the scalar/vector fields stored on the sampled volume.

This separation matters: a surprising Yamada polynomial can come from the physical
model, the discretization, the graph extraction, or the projection. Looking at each
stage makes it much easier to identify which step is responsible.

### 2.1 Build a Hopf-link non-Hermitian model

`gamma_field` controls the non-Hermitian strength in this example. The model is sampled
with `dimension=300`, meaning 300 grid points are used along each momentum direction.

The resulting `field_ske` object is the main object used throughout the next sections.
It gives access to the continuous surface, skeleton points, embedded graph, and physical
fields without rebuilding the model each time.

The built-in nodal constructors define a two-band non-Hermitian Hamiltonian from a
complex polynomial $f(z,w)$.  They first return the Bloch vector

$$
\mathbf d_f(\mathbf k)
=
\left(
\operatorname{Re}f,\,
i\gamma,\,
\operatorname{Im}f
\right),
$$

and `NodalSkeleton` constructs

$$
H_f(\mathbf k;\gamma)
=
\mathbf d_f(\mathbf k)\cdot\boldsymbol{\sigma}
=
\begin{pmatrix}
\operatorname{Im}f & \operatorname{Re}f+\gamma\\
\operatorname{Re}f-\gamma & -\operatorname{Im}f
\end{pmatrix}.
$$

Unless a model explicitly overrides them,

$$
z
=
\cos(2k_z)+c
+i\left(
\cos k_x+\cos k_y+\cos k_z-m
\right),
\qquad
w
=
\sin k_x+i\sin k_y.
$$

For the Hopf-link model,

$$
f_{\mathrm{Hopf}}(z,w)=z^2-w^2,
\qquad
c=0.5,\quad m=2.
$$

In [ ]:
gamma_field = 0.6

field_ske = NodalSkeleton(
    hopf_link_bloch_vector(
        gamma_field,
        k_symbols=(kx, ky, kz),
    ),
    k_symbols=(kx, ky, kz),
    dimension=300,
    axis_scale=(1.0, 1.0, 1.5),
)

field_surface = field_ske.exceptional_surface_pv
field_points = field_ske.skeleton_coords
field_graph = field_ske.skeleton_graph(
    simplify=True,
    smooth_epsilon=2,
)
field_volume = field_ske.fields_pv

print_surface_summary("exceptional surface", field_surface)
print("skeleton points      =", len(field_points))
print(
    "graph nodes / edges  =",
    (field_graph.number_of_nodes(), field_graph.number_of_edges()),
)
print("graph trivalent      =", field_graph.graph.get("is_trivalent"))

### 2.2 Inspect the complete exceptional surface

Always inspect the **complete extracted surface** before reducing it to a graph.

The notebook deliberately uses

```python
surface = ske.exceptional_surface_pv
```

rather than

```python
surface = ske.exceptional_surface_pv.connectivity("largest")
```

because `"largest"` discards every connected component except the largest one. That can
make a legitimate smaller component—or a numerically separated piece of a thin
surface—look as if it is missing.

The component diagnostic printed above tells you how many disconnected surface pieces
were extracted. Only select the largest component when that is an explicit part of your
physical analysis, and state that choice clearly.

A second useful view at this same stage is the **surface Brillouin-zone silhouette**:
the continuous exceptional surface is projected onto the boundary planes of the sampled
momentum region.  This is only a geometric surface visualization.  It is different from
the later planar graph projection used to build the PD code, because that graph
projection must retain over/under crossing information.

In [ ]:
show_surface(
    field_surface,
    title="Hopf-link exceptional surface",
    opacity=0.50,
).show()


# Optional surface Brillouin-zone silhouette view of the same Hopf model.
# This remains part of surface inspection; it is not the Yamada projection.
try:
    plotter = field_ske.plot_exceptional_surface(
        add_silhouettes=True,
        silh_origins=np.diag(
            [-np.pi, -np.pi, 0]
        ),
    )

    plotter.show_bounds(
        xtitle="kx",
        ytitle="ky",
        ztitle="kz",
    )

    plotter.add_bounding_box()
    plotter.show()

except TypeError as exc:
    print(
        "This installed plotting interface does not expose "
        "the `add_silhouettes` keyword. "
        f"Details: {exc}"
    )

### 2.3 Inspect the medial-axis skeleton points

These points are the discrete skeleton obtained from the sampled volume **before**
graph simplification. They are useful for checking whether the grid resolution and
skeletonization captured the geometry you intended.

In [ ]:
show_skeleton_points(
    field_points,
    title="Medial-axis skeleton extracted from the surface",
).show()

### 2.4 Inspect the simplified spatial graph

This embedded graph is the object that will later be projected and classified.

Before continuing, check that the graph has the expected branches, junctions, loops,
and connected components. Topological calculations should not be used to compensate
for an obviously incorrect extraction.

In [ ]:
show_graph(
    field_graph,
    title="Simplified embedded spatial graph",
).show()

### 2.5 Inspect the stored physical fields

`fields_pv` stores scalar and vector fields on the sampled 3D volume. The arrays used
in this notebook include:

- `real`, `imag`: real and imaginary parts of the energy;
- `gap`: band-gap magnitude;
- `ES_helper`: scalar level-set helper used for the exceptional surface;
- `im_disp`: imaginary-energy dispersion vector;
- `|im_disp|`: its magnitude;
- `berry`: Berry-curvature vector;
- `|berry|`: Berry-curvature magnitude.

Printing the available arrays is a useful first diagnostic when you adapt the workflow
to another Hamiltonian.

In [ ]:
print("Available point-data arrays:")
for name in field_volume.point_data:
    arr = np.asarray(field_volume.point_data[name])
    finite = arr[np.isfinite(arr)]
    if finite.size:
        print(
            f"{name:24s} "
            f"shape={arr.shape!s:14s} "
            f"min={finite.min(): .4g} "
            f"max={finite.max(): .4g}"
        )
    else:
        print(f"{name:24s} shape={arr.shape!s:14s} no finite values")

## 3. Berry curvature: inspect the field on and inside the exceptional surface

This section keeps the geometric and Berry-field views together.

First, select three planes through the exceptional surface. Then inspect the
Berry-curvature vector field on **those same planes**, followed by the three-dimensional
Berry-curvature arrows inside the transparent surface and the Berry-oriented graph.

Using the same planes throughout makes it much easier to compare geometry and vector
routing without changing the reference slice between figures.

### 3.1 Select planes and show their intersections with the surface

The three planes used throughout the Berry and dispersion examples are

- $k_x=0$,
- $k_y=0$, and
- $k_z=\pi/2$.

The translucent object is the **complete exceptional surface**. The colored curves are
the intersections of those three planes with that surface.

In [ ]:
# These same three planes are reused for both Berry curvature and
# imaginary-energy dispersion so that the two vector fields can be compared directly.
SELECTED_PLANES = [
    {
        "label": r"$k_x=0$",
        "axis": 0,
        "value": 0.0,
        "normal": (1, 0, 0),
        "origin": (0, 0, 0),
        "color": RED,
    },
    {
        "label": r"$k_y=0$",
        "axis": 1,
        "value": 0.0,
        "normal": (0, 1, 0),
        "origin": (0, 0, 0),
        "color": "#2ca02c",
    },
    {
        "label": r"$k_z=\pi/2$",
        "axis": 2,
        "value": np.pi / 2,
        "normal": (0, 0, 1),
        "origin": (0, 0, np.pi / 2),
        "color": "#ff7f0e",
    },
]


def _nearest_grid_index(values, target):
    values = np.asarray(values)
    return int(
        np.argmin(
            np.abs(values - target)
        )
    )


def _selected_plane_data(
    vector_grid,
    interior_mask_3d,
    skeleton,
    plane,
):
    """Extract one 2D vector-field slice with its in-plane components."""
    axis = plane["axis"]
    value = plane["value"]

    kx_values = np.asarray(skeleton.kx_vals)
    ky_values = np.asarray(skeleton.ky_vals)
    kz_values = np.asarray(skeleton.kz_vals)

    if axis == 0:
        index = _nearest_grid_index(
            kx_values,
            value,
        )
        vector_slice = vector_grid[index, :, :, :]
        mask_slice = interior_mask_3d[index, :, :]

        x_values = ky_values
        y_values = kz_values
        u = vector_slice[..., 1]
        v = vector_slice[..., 2]
        actual_value = kx_values[index]
        x_label = r"$k_y$"
        y_label = r"$k_z$"

    elif axis == 1:
        index = _nearest_grid_index(
            ky_values,
            value,
        )
        vector_slice = vector_grid[:, index, :, :]
        mask_slice = interior_mask_3d[:, index, :]

        x_values = kx_values
        y_values = kz_values
        u = vector_slice[..., 0]
        v = vector_slice[..., 2]
        actual_value = ky_values[index]
        x_label = r"$k_x$"
        y_label = r"$k_z$"

    elif axis == 2:
        index = _nearest_grid_index(
            kz_values,
            value,
        )
        vector_slice = vector_grid[:, :, index, :]
        mask_slice = interior_mask_3d[:, :, index]

        x_values = kx_values
        y_values = ky_values
        u = vector_slice[..., 0]
        v = vector_slice[..., 1]
        actual_value = kz_values[index]
        x_label = r"$k_x$"
        y_label = r"$k_y$"

    else:
        raise ValueError(
            "plane['axis'] must be 0, 1, or 2."
        )

    magnitude = np.linalg.norm(
        vector_slice,
        axis=-1,
    )

    return {
        "index": index,
        "actual_value": actual_value,
        "x_values": x_values,
        "y_values": y_values,
        "u": u,
        "v": v,
        "magnitude": magnitude,
        "mask": np.asarray(
            mask_slice,
            dtype=float,
        ),
        "x_label": x_label,
        "y_label": y_label,
    }


def plot_vector_field_on_selected_planes(
    vector_grid,
    interior_mask_3d,
    skeleton,
    *,
    title,
    colorbar_label,
    max_arrows_per_axis=18,
):
    """Plot magnitude + in-plane arrows on the three selected planes."""
    fig, axes = plt.subplots(
        1,
        len(SELECTED_PLANES),
        figsize=(16.2, 5.0),
        constrained_layout=True,
    )

    for ax, plane in zip(
        axes,
        SELECTED_PLANES,
    ):
        plane_data = _selected_plane_data(
            vector_grid,
            interior_mask_3d,
            skeleton,
            plane,
        )

        magnitude_clip, _ = nice_percentile_clip(
            plane_data["magnitude"],
            upper=0.985,
        )
        display = np.log10(
            1.0 + magnitude_clip
        )

        image = ax.imshow(
            display.T,
            origin="lower",
            extent=[
                plane_data["x_values"][0],
                plane_data["x_values"][-1],
                plane_data["y_values"][0],
                plane_data["y_values"][-1],
            ],
            aspect="equal",
        )

        # Show where the selected plane intersects the interior region.
        mask = plane_data["mask"]
        if np.any(mask > 0) and np.any(mask <= 0):
            ax.contour(
                plane_data["x_values"],
                plane_data["y_values"],
                mask.T,
                levels=[0.5],
                linewidths=1.2,
            )

        # Normalize only for arrow direction; magnitude remains in the heatmap.
        stride_x = max(
            1,
            len(plane_data["x_values"])
            // max_arrows_per_axis,
        )
        stride_y = max(
            1,
            len(plane_data["y_values"])
            // max_arrows_per_axis,
        )

        u = plane_data["u"][
            ::stride_x,
            ::stride_y,
        ]
        v = plane_data["v"][
            ::stride_x,
            ::stride_y,
        ]

        norm = np.sqrt(
            u**2 + v**2
        )
        finite = (
            np.isfinite(u)
            & np.isfinite(v)
            & (norm > 0)
        )

        u_display = np.zeros_like(
            u,
            dtype=float,
        )
        v_display = np.zeros_like(
            v,
            dtype=float,
        )

        u_display[finite] = (
            u[finite]
            / norm[finite]
        )
        v_display[finite] = (
            v[finite]
            / norm[finite]
        )

        x_quiver = plane_data["x_values"][
            ::stride_x
        ]
        y_quiver = plane_data["y_values"][
            ::stride_y
        ]

        ax.quiver(
            x_quiver,
            y_quiver,
            u_display.T,
            v_display.T,
            pivot="mid",
            angles="xy",
            scale=18,
            width=0.004,
        )

        ax.set_xlabel(
            plane_data["x_label"]
        )
        ax.set_ylabel(
            plane_data["y_label"]
        )
        ax.set_title(
            f"{plane['label']}\n"
            f"sampled at {plane_data['actual_value']:.3f}"
        )

        fig.colorbar(
            image,
            ax=ax,
            label=colorbar_label,
            fraction=0.046,
            pad=0.04,
        )

    fig.suptitle(
        title,
        fontsize=14,
    )
    plt.show()


surface_plane_slices = []

for plane in SELECTED_PLANES:
    sliced = field_surface.slice(
        normal=plane["normal"],
        origin=plane["origin"],
    )

    surface_plane_slices.append(
        (
            plane["label"],
            sliced,
            plane["color"],
        )
    )

    print(
        plane["label"],
        "points / cells =",
        (
            sliced.n_points,
            sliced.n_cells,
        ),
    )


fig = go.Figure()

mesh = field_surface.triangulate()
faces = mesh.faces.reshape(-1, 4)[:, 1:]
pts = np.asarray(mesh.points)

fig.add_trace(
    go.Mesh3d(
        x=pts[:, 0],
        y=pts[:, 1],
        z=pts[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color=BLUE,
        opacity=0.18,
        showscale=False,
        name="exceptional surface",
    )
)

for label, sliced, color in surface_plane_slices:
    for segment in polyline_traces_from_slice(
        sliced
    ):
        fig.add_trace(
            go.Scatter3d(
                x=segment[:, 0],
                y=segment[:, 1],
                z=segment[:, 2],
                mode="lines",
                line=dict(
                    color=color,
                    width=7,
                ),
                name=label,
                showlegend=False,
            )
        )

clean_scene(
    fig,
    title="Selected planes through the exceptional surface",
).show()

### 3.2 Plot Berry curvature on the same selected planes

For each plane, the heatmap shows the Berry-curvature magnitude and the arrows show the
**in-plane direction** of $\boldsymbol{\Omega}(\mathbf{k})$.

Arrow lengths are normalized for readability; use the heatmap to read relative
magnitude and the arrows to read direction.

In [ ]:
berry_grid = np.asarray(
    field_ske.berry_curvature
)

interior_mask_3d = np.asarray(
    field_ske._interior_mask,
    dtype=float,
)

plot_vector_field_on_selected_planes(
    berry_grid,
    interior_mask_3d,
    field_ske,
    title="Berry curvature on the selected planes",
    colorbar_label=r"$\log_{10}(1+|\Omega|)$",
)

### 3.3 Show Berry-curvature arrows inside the transparent surface

This view shows the **three-dimensional Berry-curvature vectors inside the exceptional
surface**.

The surface is deliberately very transparent and the arrows are deliberately large so
that the interior vector field remains visible. No spatial graph is drawn in this
figure.


In [ ]:
points = np.asarray(field_volume.points)
berry_vec = np.asarray(field_volume.point_data["berry"])
berry_strength = np.linalg.norm(berry_vec, axis=1)
es_helper = np.asarray(field_volume.point_data["ES_helper"])

# Keep only finite, nonzero Berry vectors inside the exceptional surface.
valid = (
    np.isfinite(berry_strength)
    & (berry_strength > 0)
    & np.isfinite(es_helper)
)
inside = es_helper < 0
candidate = np.flatnonzero(valid & inside)

if len(candidate) == 0:
    raise RuntimeError(
        "No finite Berry-curvature vectors were found inside the exceptional surface."
    )

# Remove very weak vectors and very large near-singular outliers.
candidate_strength = berry_strength[candidate]
low = np.quantile(candidate_strength, 0.30)
high = np.quantile(candidate_strength, 0.97)
candidate = candidate[
    (candidate_strength >= low) & (candidate_strength <= high)
]

# Use fewer arrows so each one is easier to see.
MAX_BERRY_ARROWS = 50
if len(candidate) > MAX_BERRY_ARROWS:
    take = np.linspace(0, len(candidate) - 1, MAX_BERRY_ARROWS, dtype=int)
    arrow_idx = candidate[take]
else:
    arrow_idx = candidate

arrow_strength = berry_strength[arrow_idx]
arrow_dir = berry_vec[arrow_idx] / (arrow_strength[:, None] + 1e-12)

# Make arrows MUCH longer.
strength_clip, strength_scale = nice_percentile_clip(
    arrow_strength,
    upper=0.97,
)
relative_strength = strength_clip / (strength_scale + 1e-12)

# Bigger vector lengths than before
arrow_length = 0.18 + 0.05 * relative_strength
arrow_vec = arrow_dir * arrow_length[:, None]

fig = go.Figure()

# Make the surface more transparent so arrows are visible inside it.
fig.add_trace(
    mesh_trace(
        field_surface,
        opacity=0.04,
        name="exceptional surface",
    )
)

# Show only the Berry-curvature arrows.
fig.add_trace(
    go.Cone(
        x=points[arrow_idx, 0],
        y=points[arrow_idx, 1],
        z=points[arrow_idx, 2],
        u=arrow_vec[:, 0],
        v=arrow_vec[:, 1],
        w=arrow_vec[:, 2],
        anchor="tail",
        sizemode="absolute",
        sizeref=1,   # much bigger cones
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="relative |Ω|"),
        opacity=1.0,
        name="Berry-curvature arrows",
    )
)

fig.update_layout(
    title="Berry-curvature vectors inside the exceptional surface",
    showlegend=False,
    scene=dict(
        xaxis=dict(title="k_x", showbackground=False),
        yaxis=dict(title="k_y", showbackground=False),
        zaxis=dict(title="k_z", showbackground=False),
        aspectmode="data",
        camera=dict(eye=dict(x=1.7, y=1.6, z=1.3)),
    ),
    margin=dict(l=0, r=0, b=0, t=35),
)
fig.show()

### 3.4 Orient the spatial graph using the Berry field

After inspecting the continuous Berry field, you can reduce its directional information
onto the extracted graph.

This is a different visualization from the previous figure: here the graph is the object
being oriented by the local Berry flow.


In [ ]:
# Build an oriented version of the extracted spatial graph using the local Berry field.
vol_points = np.asarray(field_volume.points)
vol_berry = np.asarray(field_volume.point_data["berry"])

fig = go.Figure()

# Light surface for context.
fig.add_trace(mesh_trace(field_surface, opacity=0.08, name="exceptional surface"))

# Plot vertices.
vx = []
vy = []
vz = []
for node, attrs in field_graph.nodes(data=True):
    pos = np.asarray(attrs.get("pos", attrs.get("position")))
    vx.append(pos[0]); vy.append(pos[1]); vz.append(pos[2])

fig.add_trace(
    go.Scatter3d(
        x=vx, y=vy, z=vz,
        mode="markers",
        marker=dict(size=4),
        name="vertices",
    )
)

arrow_x = []
arrow_y = []
arrow_z = []
arrow_u = []
arrow_v = []
arrow_w = []

def _nearest_berry_vectors(query_points):
    idx = []
    for q in query_points:
        d2 = np.sum((vol_points - q)**2, axis=1)
        idx.append(int(np.argmin(d2)))
    return vol_berry[np.array(idx, dtype=int)]

for _, _, edge_data in field_graph.edges(data=True):
    pts = edge_data.get("pts", edge_data.get("curve", None))
    if pts is None:
        continue
    pts = np.asarray(pts)
    if pts.ndim != 2 or pts.shape[0] < 2:
        continue

    mids = 0.5 * (pts[:-1] + pts[1:])
    tang = pts[1:] - pts[:-1]
    seglen = np.linalg.norm(tang, axis=1)
    mask = seglen > 1e-12
    if not np.any(mask):
        continue

    mids = mids[mask]
    tang = tang[mask]
    seglen = seglen[mask]
    tang_unit = tang / seglen[:, None]

    local_berry = _nearest_berry_vectors(mids)
    orient_score = np.sum(np.sum(local_berry * tang_unit, axis=1))

    if orient_score < 0:
        pts = pts[::-1]
        mids = 0.5 * (pts[:-1] + pts[1:])
        tang = pts[1:] - pts[:-1]
        seglen = np.linalg.norm(tang, axis=1)
        mask = seglen > 1e-12
        mids = mids[mask]
        tang = tang[mask]
        seglen = seglen[mask]
        tang_unit = tang / seglen[:, None]

    # Edge trace
    fig.add_trace(
        go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode="lines",
            line=dict(width=7),
            name="oriented edge",
            showlegend=False,
        )
    )

    # A few arrow markers along the edge orientation
    n_arrows = min(4, len(mids))
    take = np.linspace(0, len(mids) - 1, n_arrows, dtype=int)
    mids_show = mids[take]
    tang_show = tang_unit[take]
    arrow_x.extend(mids_show[:, 0])
    arrow_y.extend(mids_show[:, 1])
    arrow_z.extend(mids_show[:, 2])
    arrow_u.extend(0.10 * tang_show[:, 0])
    arrow_v.extend(0.10 * tang_show[:, 1])
    arrow_w.extend(0.10 * tang_show[:, 2])

fig.add_trace(
    go.Cone(
        x=arrow_x, y=arrow_y, z=arrow_z,
        u=arrow_u, v=arrow_v, w=arrow_w,
        sizemode="absolute",
        sizeref=0.1,
        showscale=False,
        opacity=0.85,
        name="edge orientation",
    )
)

fig.update_layout(
    title="Oriented Berry-curvature spatial graph",
    showlegend=False,
    scene=dict(
        xaxis=dict(title="k_x", showbackground=False),
        yaxis=dict(title="k_y", showbackground=False),
        zaxis=dict(title="k_z", showbackground=False),
        aspectmode="data",
        camera=dict(eye=dict(x=1.55, y=1.45, z=1.20)),
    ),
    margin=dict(l=0, r=0, b=0, t=35),
)
fig.show()

### 3.5 Build additional diagnostic fields from `fields_pv`

After inspecting the Berry-curvature field, you can derive additional scalar or vector
diagnostics from the arrays stored in `fields_pv`.

The example below computes gradients of existing field magnitudes. This is useful when
you want to locate regions where a physical field changes rapidly.

These derived arrays are **diagnostic fields**. They do not change the spatial graph
unless you explicitly use them to define a new graph-construction rule.

In [ ]:
custom_field_ske = NodalSkeleton(
    hopf_link_bloch_vector(0.40, k_symbols=(kx, ky, kz)),
    k_symbols=(kx, ky, kz),
    dimension=200,
    axis_scale=(1.0, 1.0, 1.5),
)

vol = custom_field_ske.fields_pv.copy()
available = set(vol.point_data.keys())

if "|berry|" in available:
    vol = vol.compute_derivative(scalars="|berry|", gradient="grad_abs_berry")
    grad = np.linalg.norm(vol.point_data["grad_abs_berry"], axis=-1)
    vol.point_data["log10_grad_abs_berry_plus_1"] = np.log10(grad + 1)

if "|im_disp|" in available:
    vol = vol.compute_derivative(scalars="|im_disp|", gradient="grad_abs_im_disp")
    grad = np.linalg.norm(vol.point_data["grad_abs_im_disp"], axis=-1)
    vol.point_data["log10_grad_abs_im_disp_plus_1"] = np.log10(grad + 1)

print("Available custom-field arrays:")
for name in vol.point_data.keys():
    print(" ", name)

## 4. Imaginary-energy dispersion on the same selected planes

The sampled volume also stores the imaginary-energy dispersion vector `im_disp`.

To make the comparison with Berry curvature direct, this section uses the **same three
planes** chosen above: $k_x=0$, $k_y=0$, and $k_z=\pi/2$.

Again, the heatmap encodes magnitude while normalized arrows show the in-plane routing
direction.

In [ ]:
dispersion_grid = np.asarray(
    field_volume.point_data["im_disp"]
).reshape(
    (
        field_ske.dimension,
        field_ske.dimension,
        field_ske.dimension,
        3,
    ),
    order="F",
)

plot_vector_field_on_selected_planes(
    dispersion_grid,
    interior_mask_3d,
    field_ske,
    title="Imaginary-energy dispersion on the selected planes",
    colorbar_label=r"$\log_{10}(1+|\nabla\,\mathrm{Im}E|)$",
)

## 5. Compare nodal and exceptional-surface model families

This section is the single model-gallery section for the physics notebook.  Each
representative Hamiltonian is built **once**, and the complete extracted surface and
its spatial graph are stored together for comparison.

The built-in nodal constructors use

$$
\mathbf d_f(\mathbf k)
=
\left(
\operatorname{Re}f,\,
i\gamma,\,
\operatorname{Im}f
\right),
$$

so that

$$
H_f(\mathbf k;\gamma)
=
\begin{pmatrix}
\operatorname{Im}f & \operatorname{Re}f+\gamma\\
\operatorname{Re}f-\gamma & -\operatorname{Im}f
\end{pmatrix}.
$$

Unless a model overrides them,

$$
z
=
\cos(2k_z)+c
+i\left(
\cos k_x+\cos k_y+\cos k_z-m
\right),
\qquad
w
=
\sin k_x+i\sin k_y.
$$

| Model | $f(z,w)$ |
|---|---|
| Unknot | $z-w$ |
| Hopf link | $z^2-w^2$ |
| Trefoil | $z^2-w^3$ |
| Solomon link | $z^2-w^4$ |
| Three-link | $z(z^2-w^2)$ |
| Torus $(p,q)$ | $z^p-w^q$ |
| Awesome | $z(z^2-w^4+w)$ |

The surface gallery and graph gallery are shown as **separate figures**, so a
semi-transparent surface cannot obscure its extracted graph.  The gallery models use
`dimension=300`.

In [ ]:
model_gallery_specs = [
    (
        "Unknot",
        lambda: unknot_bloch_vector(
            0.10,
            k_symbols=(kx, ky, kz),
        ),
    ),
    (
        "Hopf link",
        lambda: hopf_link_bloch_vector(
            0.20,
            k_symbols=(kx, ky, kz),
        ),
    ),
    (
        "Trefoil",
        lambda: trefoil_bloch_vector(
            0.25,
            k_symbols=(kx, ky, kz),
        ),
    ),
    (
        "Solomon",
        lambda: solomon_bloch_vector(
            1.00,
            k_symbols=(kx, ky, kz),
        ),
    ),
    (
        "Three-link",
        lambda: threelink_bloch_vector(
            0.41,
            k_symbols=(kx, ky, kz),
        ),
    ),
    (
        "Torus (1,2)",
        lambda: pq_torus_knot_bloch_vector(
            1,
            2,
            0.50,
            k_symbols=(kx, ky, kz),
        ),
    ),
    (
        "Torus (3,7)",
        lambda: pq_torus_knot_bloch_vector(
            3,
            7,
            0.20,
            k_symbols=(kx, ky, kz),
            c=0.7,
            m=2.0,
        ),
    ),
    (
        "Awesome",
        lambda: awesome_bloch_vector(
            0.20,
            k_symbols=(kx, ky, kz),
            c=0.5,
        ),
    ),
]


model_gallery_records = []

for name, builder in model_gallery_specs:
    ske_model = NodalSkeleton(
        builder(),
        k_symbols=(kx, ky, kz),
        dimension=300,
        axis_scale=(1.0, 1.0, 1.5),
    )

    surface_model = (
        ske_model.exceptional_surface_pv
    )

    graph_model = None
    graph_error = None
    is_planar = None

    try:
        graph_model = ske_model.skeleton_graph(
            simplify=True,
            smooth_epsilon=2,
        )

        is_planar = nx.check_planarity(
            nx.Graph(graph_model)
        )[0]

    except Exception as exc:
        graph_error = (
            f"{type(exc).__name__}: {exc}"
        )

    record = dict(
        name=name,
        skeleton=ske_model,
        surface=surface_model,
        graph=graph_model,
        graph_error=graph_error,
        is_planar=is_planar,
    )

    model_gallery_records.append(
        record
    )

    print(name)
    print_surface_summary(
        "  full surface",
        surface_model,
    )

    if graph_model is None:
        print(
            "  graph_stage =",
            graph_error,
        )
    else:
        print(
            "  graph_nodes_edges =",
            (
                graph_model.number_of_nodes(),
                graph_model.number_of_edges(),
            ),
        )
        print(
            "  abstract_planar =",
            is_planar,
        )


# ------------------------------------------------------------
# Surface gallery
# ------------------------------------------------------------

fig = make_subplots(
    rows=2,
    cols=4,
    specs=[
        [
            {"type": "scene"}
            for _ in range(4)
        ],
        [
            {"type": "scene"}
            for _ in range(4)
        ],
    ],
    subplot_titles=[
        record["name"]
        for record in model_gallery_records
    ],
    horizontal_spacing=0.01,
    vertical_spacing=0.03,
)

for index, record in enumerate(
    model_gallery_records
):
    row = 1 if index < 4 else 2
    col = index % 4 + 1

    add_surface_trace(
        fig,
        record["surface"],
        row=row,
        col=col,
        opacity=0.46,
    )

style_plotly_scenes(
    fig,
    8,
    width=1120,
    height=760,
).show()


# ------------------------------------------------------------
# Extracted-graph gallery
# ------------------------------------------------------------

fig = make_subplots(
    rows=2,
    cols=4,
    specs=[
        [
            {"type": "scene"}
            for _ in range(4)
        ],
        [
            {"type": "scene"}
            for _ in range(4)
        ],
    ],
    subplot_titles=[
        record["name"]
        for record in model_gallery_records
    ],
    horizontal_spacing=0.01,
    vertical_spacing=0.03,
)

for index, record in enumerate(
    model_gallery_records
):
    graph_model = record["graph"]

    if graph_model is None:
        continue

    row = 1 if index < 4 else 2
    col = index % 4 + 1

    add_graph_traces(
        fig,
        graph_model,
        row=row,
        col=col,
    )

style_plotly_scenes(
    fig,
    8,
    width=1120,
    height=760,
).show()

## 6. Scan the non-Hermitian strength

Use a parameter sweep when you want to know **where the extracted topology changes as a
physical control parameter changes**.

The example varies `gamma`. For every value it keeps the continuous surface and embedded
graph together. When adapting this section, change the tuple of `gamma` values and
compare both objects before interpreting a change in the polynomial.

A topology change should be supported by a visible and numerically credible change in
the extracted geometry, not only by a different final string.

The top row shows the **full** extracted surface for each $\gamma$. The title reports
the number of connected surface components. This is important near thin or nearly
pinched geometries: selecting only the largest component could silently remove a
smaller piece and make the surface look incomplete.

The sweep uses the same Hopf-link Hamiltonian for every panel,

$$
H_{\mathrm{Hopf}}(\mathbf k;\gamma)
=
\begin{pmatrix}
\operatorname{Im}(z^2-w^2) &
\operatorname{Re}(z^2-w^2)+\gamma\\
\operatorname{Re}(z^2-w^2)-\gamma &
-\operatorname{Im}(z^2-w^2)
\end{pmatrix},
$$

and varies only $\gamma$.

In [ ]:
hopf_gamma_records = []

for gamma in (0.15, 0.20, 0.50):
    ske_gamma = NodalSkeleton(
        hopf_link_bloch_vector(
            gamma,
            k_symbols=(kx, ky, kz),
        ),
        k_symbols=(kx, ky, kz),
        dimension=200,
        axis_scale=(1.0, 1.0, 1.5),
    )

    # Keep the COMPLETE extracted surface.
    surface_gamma = ske_gamma.exceptional_surface_pv
    n_components, component_sizes = surface_component_summary(surface_gamma)

    graph_gamma = ske_gamma.skeleton_graph(
        simplify=True,
        smooth_epsilon=2,
    )

    hopf_gamma_records.append(
        (
            gamma,
            surface_gamma,
            graph_gamma,
            n_components,
            component_sizes,
        )
    )

    print(f"\ngamma={gamma:.2f}")
    print_surface_summary("  full surface", surface_gamma)
    print(
        "  graph nodes / edges =",
        (
            graph_gamma.number_of_nodes(),
            graph_gamma.number_of_edges(),
        ),
    )


surface_titles = []
graph_titles = []

for gamma, _, _, n_components, _ in hopf_gamma_records:
    if n_components is None:
        component_label = "components: ?"
    else:
        component_label = f"components: {n_components}"

    surface_titles.append(
        f"surface: gamma={gamma:.2f}<br>{component_label}"
    )
    graph_titles.append(
        f"graph: gamma={gamma:.2f}"
    )


fig = make_subplots(
    rows=2,
    cols=len(hopf_gamma_records),
    specs=[
        [{"type": "scene"} for _ in hopf_gamma_records],
        [{"type": "scene"} for _ in hopf_gamma_records],
    ],
    subplot_titles=surface_titles + graph_titles,
    horizontal_spacing=0.01,
    vertical_spacing=0.05,
)

for col, (
    gamma,
    surface_gamma,
    graph_gamma,
    n_components,
    component_sizes,
) in enumerate(hopf_gamma_records, start=1):

    add_surface_trace(
        fig,
        surface_gamma,
        row=1,
        col=col,
        opacity=0.46,
    )

    add_graph_traces(
        fig,
        graph_gamma,
        row=2,
        col=col,
    )

style_plotly_scenes(
    fig,
    2 * len(hopf_gamma_records),
    width=1080,
    height=760,
).show()

## 7. Compare Yamada results across models and parameter values

This section evaluates several model families at several values of the non-Hermitian
parameter $\gamma$.

For each case it computes the embedded graph, chooses a projection, and evaluates the
Yamada polynomial. The results are summarized in one compact table containing the numerical/topological
diagnostics and polynomial.  The graphs are not redrawn here because representative
model geometries already appear in Section 5.

The primary gallery calculations use `dimension=300`; targeted secondary examples use
the resolution stated in their code cells (including `dimension=200` where noted).

The built-in nodal constructors define a two-band non-Hermitian Hamiltonian from a
complex polynomial $f(z,w)$.  They first return the Bloch vector

$$
\mathbf d_f(\mathbf k)
=
\left(
\operatorname{Re}f,\,
i\gamma,\,
\operatorname{Im}f
\right),
$$

and `NodalSkeleton` constructs

$$
H_f(\mathbf k;\gamma)
=
\mathbf d_f(\mathbf k)\cdot\boldsymbol{\sigma}
=
\begin{pmatrix}
\operatorname{Im}f & \operatorname{Re}f+\gamma\\
\operatorname{Re}f-\gamma & -\operatorname{Im}f
\end{pmatrix}.
$$

Unless a model explicitly overrides them,

$$
z
=
\cos(2k_z)+c
+i\left(
\cos k_x+\cos k_y+\cos k_z-m
\right),
\qquad
w
=
\sin k_x+i\sin k_y.
$$

The model-specific complex polynomials used in these examples are

| Model | $f(z,w)$ |
|---|---|
| Unknot | $z-w$ |
| Hopf link | $z^2-w^2$ |
| Trefoil | $z^2-w^3$ |
| Solomon link | $z^2-w^4$ |
| Three-link | $z(z^2-w^2)$ |
| Torus $(p,q)$ | $z^p-w^q$ |
| “Awesome” | $z(z^2-w^4+w)$ |

The Solomon constructor uses $c=0.333$ and $m=2$ by default.  Torus examples can
override $c$ and $m$ explicitly, as shown in the corresponding code.

In [ ]:
yamada_table_specs = [
    (
        "Hopf link",
        hopf_link_bloch_vector,
        [0.10, 0.20, 0.50],
    ),
    (
        "Trefoil",
        trefoil_bloch_vector,
        [0.10, 0.19, 0.25],
    ),
    (
        "Torus (1,2)",
        lambda gamma, k_symbols: pq_torus_knot_bloch_vector(
            1,
            2,
            gamma,
            k_symbols=k_symbols,
        ),
        [0.12, 0.50, 0.70],
    ),
    (
        "Solomon",
        solomon_bloch_vector,
        [0.12, 1.00, 2.00],
    ),
]

yamada_rows = []

for family_index, (
    family,
    builder,
    gammas,
) in enumerate(yamada_table_specs):

    for gamma_index, gamma in enumerate(gammas):
        row = {
            "family": family,
            "gamma": gamma,
            "dimension": 200,
            "status": "ok",
            "graph": None,
            "family_index": family_index,
            "gamma_index": gamma_index,
        }

        try:
            ske_row = NodalSkeleton(
                builder(
                    gamma,
                    k_symbols=(kx, ky, kz),
                ),
                k_symbols=(kx, ky, kz),
                dimension=200,
                axis_scale=(1.0, 1.0, 1.5),
            )

            graph_row = ske_row.skeleton_graph(
                simplify=True,
                smooth_epsilon=2,
            )

            projection_row = select_projection(
                graph_row,
                num_rotation_samples=8,
            )

            polynomial_row = compute_yamada_polynomial(
                graph_row,
                Y,
                rotation_angles=projection_row.rotation_angles,
                n_jobs=1,
            )

            row.update(
                graph=graph_row,
                nodes=graph_row.number_of_nodes(),
                edges=graph_row.number_of_edges(),
                crossings=projection_row.num_crossings,
                upsilon=str(
                    sp.expand(polynomial_row)
                ),
            )

        except Exception as exc:
            row.update(
                status="inspect",
                reason=(
                    f"{type(exc).__name__}: "
                    f"{str(exc)[:120]}"
                ),
                nodes="—",
                edges="—",
                crossings="—",
                upsilon="—",
            )

        yamada_rows.append(row)


# ----- Visible result 1: compact table -----
table_fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=[
                    "<b>Family</b>",
                    "<b>gamma</b>",
                    "<b>Nodes</b>",
                    "<b>Edges</b>",
                    "<b>Crossings</b>",
                    "<b>Computed Upsilon(G;Y)</b>",
                    "<b>Status</b>",
                ],
                align="left",
            ),
            cells=dict(
                values=[
                    [
                        row["family"]
                        for row in yamada_rows
                    ],
                    [
                        row["gamma"]
                        for row in yamada_rows
                    ],
                    [
                        row["nodes"]
                        for row in yamada_rows
                    ],
                    [
                        row["edges"]
                        for row in yamada_rows
                    ],
                    [
                        row["crossings"]
                        for row in yamada_rows
                    ],
                    [
                        row["upsilon"]
                        for row in yamada_rows
                    ],
                    [
                        row["status"]
                        for row in yamada_rows
                    ],
                ],
                align="left",
                height=28,
            ),
            columnwidth=[
                100,
                55,
                50,
                50,
                70,
                320,
                75,
            ],
        )
    ]
)

table_fig.update_layout(
    title="Model-level Yamada results",
    height=570,
    margin=dict(
        l=10,
        r=10,
        t=45,
        b=10,
    ),
)
table_fig.show()

## 8. Material Fermi-surface workflows

The material examples below keep the **continuous surface** and its **extracted graph**
as two consecutive standalone figures.  This avoids squeezing both objects into a
single top/bottom subplot and makes their geometry easier to inspect.

Each material is passed to the current `MaterialFermiSurface` API through a SymPy
Bloch Hamiltonian, a selected `band_pair`, a momentum-space `span`, and a `gap_tol`.
The examples intentionally show the complete extracted surface rather than silently
selecting only its largest connected component.

Graph cleanup is applied through the shared `build_material_graph(...)` helper.  The
individual operations—leaf removal, degree-2-chain collapse, short-edge contraction,
and smoothing—are explained once in **[2. Core Workflows](../02_core_workflows.ipynb)**
instead of being repeated for every material.

All material plots use the same standalone view with a farther camera and no titles.

### 8.1 Load the material Hamiltonians and surface class

In [ ]:
from knotted_graph.applications.materials import (
    TI3AL_PARAMETERS,
    D6_PARAMETERS,
    YH3_PARAMETERS,
    H_Ti3Al_sympy,
    H_D6_sympy,
    H_YH3_sympy,
    MaterialFermiSurface,
)

from knotted_graph.core import (
    contract_short_edges,
    remove_leaf_nodes,
    simplify_edges,
    smooth_edges,
)


H_ti3al = H_Ti3Al_sympy(
    k_symbols=(kx, ky, kz)
)

H_tib2 = H_D6_sympy(
    k_symbols=(kx, ky, kz)
)

H_yh3 = H_YH3_sympy(
    k_symbols=(kx, ky, kz)
)


print(
    "Ti3Al matrix shape =",
    H_ti3al.shape,
)
print(
    "TiB2 / D6 shape    =",
    H_tib2.shape,
)
print(
    "YH3 matrix shape   =",
    H_yh3.shape,
)


def build_material_graph(
    material_skeleton,
    *,
    smooth_epsilon=2,
    remove_leaves=False,
    min_edge_length=None,
):
    """
    Build a cleaned embedded graph from a MaterialFermiSurface object.

    The detailed cleanup operations are explained once in
    02_core_workflows.ipynb.  This application-level helper applies
    the same sequence without repeating every intermediate plot:

        raw graph
        -> optional leaf removal
        -> collapse degree-2 chains
        -> optional short-edge contraction
        -> collapse any newly created degree-2 chains
        -> smooth edge polylines
    """

    raw_graph = (
        material_skeleton.skeleton_graph(
            simplify=False,
            smooth_epsilon=0,
            force_small_edge_contraction=False,
        )
    )

    if (
        raw_graph.number_of_nodes() == 0
        or raw_graph.number_of_edges() == 0
    ):
        raise RuntimeError(
            "The raw material skeleton graph is empty. "
            "Inspect gap_tol, band_pair, span, and grid resolution."
        )

    graph = raw_graph.copy()

    if remove_leaves:
        graph = remove_leaf_nodes(
            graph
        )

    graph = simplify_edges(
        graph
    )

    if min_edge_length is not None:
        graph = contract_short_edges(
            graph,
            min_length=min_edge_length,
        )

        graph = simplify_edges(
            graph
        )

    graph = smooth_edges(
        graph,
        epsilon=smooth_epsilon,
        copy=True,
    )

    return raw_graph, graph

### 8.2 $Ti_3Al$

The supplied two-band Hamiltonian is

$$
H_{Ti_3Al}(\mathbf{k})
=
\begin{pmatrix}
\varepsilon(\mathbf{k}) & 0\\
0 & -\varepsilon(\mathbf{k})
\end{pmatrix},
$$

where

$$
\varepsilon
=
\frac{1}{2}
\left[
h_1+h_2+
\sqrt{(h_1-h_2)^2+h^2}
\right],
$$

$$
h_1
=
A_1(k_x^2+k_y^2)
+B_1k_z^2+M_1,
$$

$$
h_2
=
A_2(k_x^2+k_y^2)
+B_2k_z^2+M_2,
\qquad
h=2Ck_z.
$$

The next cell prints the numerical parameters.  This example uses
`band_pair=(0, 1)` and `gap_tol=0.3`.

In [ ]:
print("Ti3Al parameters:")
for key, value in TI3AL_PARAMETERS.items():
    print(f"  {key:>3s} = {value}")

print("\nHamiltonian:")
H_ti3al

In [ ]:
ti3al_ske = MaterialFermiSurface(
    H_ti3al,
    k_symbols=(kx, ky, kz),
    span=(
        (-np.pi, np.pi),
        (-np.pi, np.pi),
        (-np.pi, np.pi),
    ),
    dimension=400,
    band_pair=(0, 1),
    gap_tol=0.3,
    force_small_edge_contraction=False,
    small_edge_limit=np.pi * 0.1,
)


ti3al_surface = (
    ti3al_ske.exceptional_surface_pv
)

print_surface_summary(
    "$Ti_3Al$ full surface",
    ti3al_surface,
)

show_material_surface(
    ti3al_surface,
    opacity=0.50,
).show()

In [ ]:
ti3al_raw_graph, ti3al_graph = (
    build_material_graph(
        ti3al_ske,
        smooth_epsilon=2,
    )
)

print(
    "raw graph nodes / edges =",
    (
        ti3al_raw_graph.number_of_nodes(),
        ti3al_raw_graph.number_of_edges(),
    ),
)

print(
    "simplified graph nodes / edges =",
    (
        ti3al_graph.number_of_nodes(),
        ti3al_graph.number_of_edges(),
    ),
)

show_material_graph(
    ti3al_graph
).show()

### 8.3 $TiB_2$ / $D_6$ model

The supplied three-band Hamiltonian is

$$
H_{D_6}(\mathbf{k})
=
\begin{pmatrix}
q_1 & h_{12} & h_{13}\\
h_{12}^{*} & q_1 & h_{23}\\
h_{13}^{*} & h_{23}^{*} & q_2
\end{pmatrix},
$$

where $k_{\pm}=k_x\pm i k_y$, $k_{xy}^2=k_x^2+k_y^2$, and

$$
q_1
=
F_1+A_1k_{xy}^2+B_1k_z^2,
$$

$$
q_2
=
F_2+A_2k_{xy}^2+B_2k_z^2
+L(k_{xy}^2)^2
+M(k_+^6+k_-^6),
$$

$$
h_{12}
=
Ck_-^2+Fk_+^4,
\qquad
h_{13}
=
Dk_-k_z,
\qquad
h_{23}
=
Dk_+k_z.
$$

The next cell prints the numerical parameters.  This example uses
`band_pair=(0, 1)` and `gap_tol=0.35`.

In [ ]:
print("TiB2 / D6 parameters:")
for key, value in D6_PARAMETERS.items():
    print(f"  {key:>3s} = {value}")

print("\nSelected matrix elements:")
print("H[0,0] =", sp.simplify(H_tib2[0, 0]))
print("H[0,1] =", sp.simplify(H_tib2[0, 1]))
print("H[0,2] =", sp.simplify(H_tib2[0, 2]))
print("H[2,2] =", sp.simplify(H_tib2[2, 2]))

In [ ]:
tib2_ske = MaterialFermiSurface(
    H_tib2,
    k_symbols=(kx, ky, kz),
    span=(
        (-1.5, 1.5),
        (-1.5, 1.5),
        (-1.5, 1.5),
    ),
    dimension=300,
    band_pair=(0, 1),
    gap_tol=0.35,
    force_small_edge_contraction=True,
    small_edge_limit=np.pi * 0.1,
)


tib2_surface = (
    tib2_ske.exceptional_surface_pv
)

print_surface_summary(
    "$TiB_2$ full surface",
    tib2_surface,
)

show_material_surface(
    tib2_surface,
    opacity=0.50,
).show()

In [ ]:
tib2_raw_graph, tib2_graph = (
    build_material_graph(
        tib2_ske,
        smooth_epsilon=2,
        min_edge_length=np.pi * 0.1,
    )
)

print(
    "raw graph nodes / edges =",
    (
        tib2_raw_graph.number_of_nodes(),
        tib2_raw_graph.number_of_edges(),
    ),
)

print(
    "simplified graph nodes / edges =",
    (
        tib2_graph.number_of_nodes(),
        tib2_graph.number_of_edges(),
    ),
)

show_material_graph(
    tib2_graph
).show()

### 8.4 $YH_3$

The supplied two-band effective Hamiltonian is

$$
H_{YH_3}(\mathbf{k})
=
\begin{pmatrix}
E(\mathbf{k}) & 0\\
0 & -E(\mathbf{k})
\end{pmatrix},
$$

with

$$
E
=
\sqrt{
(g_1^2+h_1^2)
(g_2^2+h_2^2)
(g_3^2+h_3^2)
},
$$

$$
g_1=\sin k_z,
\qquad
g_2=\sin k_x,
\qquad
g_3=\sin k_y,
$$

$$
h_1
=
a_1
\left(
r_1\cos^{n_1}k_x
+s_1\cos^{n_1}k_y
+t_1\cos^{n_1}k_z
-m_1
\right),
$$

$$
h_2
=
a_2
(\cos k_x+\cos k_y+\cos k_z-m_2),
$$

$$
h_3
=
a_3
(\cos k_x+\cos k_y+\cos k_z-m_3).
$$

The next cell constructs the current `MaterialFermiSurface` example directly from
the supplied $YH_3$ Hamiltonian.  It uses `band_pair=(0, 1)` and
`gap_tol=0.003` on a $300^3$ momentum grid.

In [ ]:
from knotted_graph.applications.materials import (
    H_YH3_sympy,
    MaterialFermiSurface,
)


# ------------------------------------------------------------
# YH3 Hamiltonian
# ------------------------------------------------------------

H_yh3 = H_YH3_sympy(
    k_symbols=(kx, ky, kz),
)


# ------------------------------------------------------------
# YH3 material surface object
# ------------------------------------------------------------

yh3_ske = MaterialFermiSurface(
    H_yh3,
    k_symbols=(kx, ky, kz),
    span=(
        (-np.pi, np.pi),
        (-np.pi, np.pi),
        (-np.pi, np.pi),
    ),
    dimension=300,
    band_pair=(0, 1),
    gap_tol=0.003,
    force_small_edge_contraction=False,
    small_edge_limit=np.pi * 0.1,
)


# ------------------------------------------------------------
# Surface
# ------------------------------------------------------------

yh3_surface = yh3_ske.exceptional_surface_pv


print_surface_summary(
    "$YH_3$ full surface",
    yh3_surface,
)


show_material_surface(
    yh3_surface,
    opacity=0.50,
).show()

In [ ]:
# ------------------------------------------------------------
# YH3 extracted graph
# ------------------------------------------------------------

yh3_raw_graph, yh3_graph = build_material_graph(
    yh3_ske,
    smooth_epsilon=2,
)


print(
    "raw graph nodes / edges =",
    (
        yh3_raw_graph.number_of_nodes(),
        yh3_raw_graph.number_of_edges(),
    ),
)


print(
    "simplified graph nodes / edges =",
    (
        yh3_graph.number_of_nodes(),
        yh3_graph.number_of_edges(),
    ),
)


print(
    "degree sequence =",
    sorted(
        dict(
            yh3_graph.degree()
        ).values()
    ),
)


show_material_graph(
    yh3_graph
).show()

### 8.5 $Co_2MnGa$

The notebook supplies a six-band Hermitian Hamiltonian

$$
H_{Co_2MnGa}
=
\begin{pmatrix}
D_d & V_{dp}\\
V_{dp}^{\dagger} & D_p
\end{pmatrix},
$$

with

$$
D_d
=
\operatorname{diag}
(\xi_{d1},\xi_{d2},\xi_{d3}),
$$

$$
D_p
=
\begin{pmatrix}
\xi_{p1} & \xi_{p12} & \xi_{p13}\\
\xi_{p12} & \xi_{p2} & \xi_{p23}\\
\xi_{p13} & \xi_{p23} & \xi_{p3}
\end{pmatrix},
$$

$$
V_{dp}
=
\begin{pmatrix}
\xi_{dp11} & 0 & \xi_{dp13}\\
\xi_{dp21} & \xi_{dp22} & 0\\
0 & \xi_{dp32} & \xi_{dp33}
\end{pmatrix}.
$$

Using $c_\mu=\cos(k_\mu/2)$ and $s_\mu=\sin(k_\mu/2)$,

$$
\begin{aligned}
\xi_{d1}
&=
4t_1c_xc_z
+2t_2(\cos k_x+\cos k_z)
+2t_3\cos k_y+\varepsilon_d,\\
\xi_{d2}
&=
4t_1c_xc_y
+2t_2(\cos k_x+\cos k_y)
+2t_3\cos k_z+\varepsilon_d,\\
\xi_{d3}
&=
4t_1c_yc_z
+2t_2(\cos k_y+\cos k_z)
+2t_3\cos k_x+\varepsilon_d,\\
\xi_{p1}
&=
4t_4c_yc_z
+2t_5(\cos k_y+\cos k_z)
+2t_6\cos k_x+\varepsilon_p,\\
\xi_{p2}
&=
4t_4c_xc_z
+2t_5(\cos k_x+\cos k_z)
+2t_6\cos k_y+\varepsilon_p,\\
\xi_{p3}
&=
4t_4c_xc_y
+2t_5(\cos k_x+\cos k_y)
+2t_6\cos k_z+\varepsilon_p.
\end{aligned}
$$

The remaining couplings are

$$
\xi_{p12}
=
-4t_7s_xs_y,
\qquad
\xi_{p13}
=
-4t_7s_xs_z,
\qquad
\xi_{p23}
=
-4t_7s_ys_z,
$$

$$
\xi_{dp11}=2t_8s_z,
\quad
\xi_{dp13}=2t_8s_x,
\quad
\xi_{dp21}=2t_8s_y,
\quad
\xi_{dp22}=2t_8s_z,
\quad
\xi_{dp32}=2t_8s_x,
\quad
\xi_{dp33}=2t_8s_y.
$$

The next cell lists the numerical hopping and onsite parameters.

In [ ]:
co2mnga_parameters = dict(
    t1=-0.31,
    t2=-0.018,
    t3=-0.01,
    t4=0.20,
    t5=-0.02,
    t6=0.04,
    t7=0.28,
    t8=-0.34,
    epsilon_d=-0.60,
    epsilon_p=0.60,
)

for key, value in co2mnga_parameters.items():
    print(f"{key:>9s} = {value}")

In [ ]:
H_co2mnga = (
    co2mnga_hamiltonian_sympy()
)

co2mnga_ske = MaterialFermiSurface(
    H_co2mnga,
    k_symbols=(kx, ky, kz),
    span=(
        (-2 * np.pi, 2 * np.pi),
        (-2 * np.pi, 2 * np.pi),
        (-2 * np.pi, 2 * np.pi),
    ),
    dimension=300,
    band_pair=(0, 1),
    gap_tol=0.3,
    force_small_edge_contraction=True,
    small_edge_limit=np.pi * 0.1,
)


co2mnga_surface = (
    co2mnga_ske.exceptional_surface_pv
)

print_surface_summary(
    "$Co_2MnGa$ full surface",
    co2mnga_surface,
)

show_material_surface(
    co2mnga_surface,
    opacity=0.50,
).show()

In [ ]:
co2mnga_raw_graph, co2mnga_graph = (
    build_material_graph(
        co2mnga_ske,
        smooth_epsilon=2,
        remove_leaves=True,
        min_edge_length=np.pi * 0.1,
    )
)


print(
    "raw graph nodes / edges =",
    (
        co2mnga_raw_graph.number_of_nodes(),
        co2mnga_raw_graph.number_of_edges(),
    ),
)


print(
    "final graph nodes / edges =",
    (
        co2mnga_graph.number_of_nodes(),
        co2mnga_graph.number_of_edges(),
    ),
)


print(
    "final degree sequence =",
    sorted(
        dict(
            co2mnga_graph.degree()
        ).values()
    ),
)


show_material_graph(
    co2mnga_graph
).show()

### 8.6 Using your own material or metamaterial Hamiltonian

The same workflow can be applied to a user-defined material or metamaterial once its
Bloch Hamiltonian is written as a SymPy matrix,

$$
H(\mathbf{k})=H(k_x,k_y,k_z).
$$

For example, a two-band model can be written as

```python
H_custom = sp.Matrix(
    [
        [h11, h12],
        [sp.conjugate(h12), h22],
    ]
)
```

where `h11`, `h12`, and `h22` are symbolic functions of `kx`, `ky`, and `kz`.

Pass the Hamiltonian directly to `MaterialFermiSurface`:

```python
custom_ske = MaterialFermiSurface(
    H_custom,
    k_symbols=(kx, ky, kz),
    span=(
        (-np.pi, np.pi),
        (-np.pi, np.pi),
        (-np.pi, np.pi),
    ),
    dimension=300,
    band_pair=(0, 1),
    gap_tol=0.3,
    force_small_edge_contraction=False,
    small_edge_limit=np.pi * 0.1,
)
```

Here, `band_pair=(i, j)` selects the two bands whose pairwise gap is used to define
the three-dimensional region, while `gap_tol` sets the corresponding threshold.

Inspect the complete surface first:

```python
custom_surface = custom_ske.exceptional_surface_pv

print_surface_summary(
    "custom material full surface",
    custom_surface,
)

show_material_surface(
    custom_surface,
    opacity=0.50,
).show()
```

Then build the embedded graph with the same application-level cleanup helper used by
the material examples:

```python
custom_raw_graph, custom_graph = (
    build_material_graph(
        custom_ske,
        smooth_epsilon=2,
        remove_leaves=False,
        min_edge_length=None,
    )
)

show_material_graph(
    custom_graph
).show()
```

If your application requires leaf removal or contraction of short numerical edges,
enable those options explicitly:

```python
custom_raw_graph, custom_graph = (
    build_material_graph(
        custom_ske,
        smooth_epsilon=2,
        remove_leaves=True,
        min_edge_length=np.pi * 0.1,
    )
)
```

The individual cleanup stages—leaf removal, degree-2-chain collapse, short-edge
contraction, and smoothing—are explained once in
**[2. Core Workflows](../02_core_workflows.ipynb)** rather than duplicated here.

After the embedded graph has been validated, continue with the common topology pipeline,

$$
H(\mathbf{k})
\longrightarrow
\text{surface}
\longrightarrow
\text{spatial graph}
\longrightarrow
\text{planar projection}
\longrightarrow
\text{PD code}
\longrightarrow
\text{Yamada polynomial}.
$$